In [ ]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score


In [ ]:
train_input = "new_data/mover_epic_final_train_features.csv"
test_input = "new_data/mover_epic_final_test_features.csv"

print("[XGBoost Stable Final] Loading datasets...")

train_df = pd.read_csv(train_input, encoding="utf-8")
test_df = pd.read_csv(test_input, encoding="utf-8")


In [ ]:
if "IN_OR_DTTM" in train_df.columns:
    train_df["IN_OR_DTTM"] = pd.to_datetime(train_df["IN_OR_DTTM"], errors="coerce")
    train_df = train_df.sort_values("IN_OR_DTTM").reset_index(drop=True)


In [ ]:
scaler_features = [
    'AGE', 'HEIGHT', 'WEIGHT',
    'SCHEDULED_START_HOUR',
    'SURGERY_DAY_OF_WEEK',
    'SURGERY_MONTH'
]

cyclical_features = [
    'DOW_SIN', 'DOW_COS',
    'MONTH_SIN', 'MONTH_COS',
    'HOUR_SIN', 'HOUR_COS'
]

binary_features = [
    'SEX_CODE',
    'ICU_ADMIN_FLAG_CODE',
    'IS_MORNING_CASE_CODE',
    'Is_Hypertension',
    'Is_Diabetes',
    'Is_Cardiac'
]

categorical_features = ['ASA_RATING_C']

procedure_cols = [c for c in train_df.columns if c.startswith("PRIMARY_PROCEDURE_NM_")]

feature_cols = (
    scaler_features +
    cyclical_features +
    binary_features +
    categorical_features +
    procedure_cols
)


In [ ]:
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df["ACTUAL_DURATION"].values
y_test = test_df["ACTUAL_DURATION"].values

y_log = np.log1p(y_train)

In [ ]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.replace("[", "_", regex=False)
        .str.replace("]", "_", regex=False)
        .str.replace("<", "_lt_", regex=False)
        .str.replace(">", "_gt_", regex=False)
    )
    return df

X_train = clean_columns(X_train)
X_test = clean_columns(X_test)

In [ ]:
print("[XGBoost Stable Final] Running TimeSeriesSplit CV...")

tscv = TimeSeriesSplit(n_splits=5)

results = {"mae": [], "rmse": [], "medae": [], "r2": []}

for fold, (tr, val) in enumerate(tscv.split(X_train), 1):

    X_tr, X_val = X_train.iloc[tr], X_train.iloc[val]
    y_tr, y_val = y_log[tr], y_log[val]

    model = xgb.XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_weight=5,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    # SAFE FIT (NO callbacks, NO early stopping issues)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    pred = np.expm1(model.predict(X_val))
    true = np.expm1(y_val)

    results["mae"].append(mean_absolute_error(true, pred))
    results["rmse"].append(np.sqrt(mean_squared_error(true, pred)))
    results["medae"].append(median_absolute_error(true, pred))
    results["r2"].append(r2_score(true, pred))

    print(f"[Fold {fold}] MAE={results['mae'][-1]:.2f} | R2={results['r2'][-1]:.4f}")


In [ ]:
print("\n" + "="*70)
print("CROSS VALIDATION SUMMARY")
print("="*70)
print(f"MAE   : {np.mean(results['mae']):.2f} ± {np.std(results['mae']):.2f}")
print(f"RMSE  : {np.mean(results['rmse']):.2f} ± {np.std(results['rmse']):.2f}")
print(f"MedAE : {np.mean(results['medae']):.2f} ± {np.std(results['medae']):.2f}")
print(f"R2    : {np.mean(results['r2']):.4f} ± {np.std(results['r2']):.4f}")


In [ ]:
print("\nTraining final model...")

final_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=5,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_log, verbose=False)


In [ ]:
pred_test = np.expm1(final_model.predict(X_test))

print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)
print(f"MAE   : {mean_absolute_error(y_test, pred_test):.2f}")
print(f"RMSE  : {np.sqrt(mean_squared_error(y_test, pred_test)):.2f}")
print(f"MedAE : {median_absolute_error(y_test, pred_test):.2f}")
print(f"R2    : {r2_score(y_test, pred_test):.4f}")
print("="*60)